In [1]:
import torch
import torch.nn as nn
import numpy as np
import sys
from torch.utils.data import TensorDataset, DataLoader

In [2]:
# Inputs: temperature, rainfall, humidity
x_train = np.array([
    [73, 67, 43], [91, 88, 64], [87, 134, 58],
    [102, 43, 37], [69, 96, 70], [73, 67, 43],
    [91, 88, 64], [87, 134, 58], [102, 43, 37],
    [69, 96, 70], [73, 67, 43], [91, 88, 64],
    [87, 134, 58], [102, 43, 37], [69, 96, 70]],
    dtype='float32')

# Targets: apples, oranges (yield)
y_train = np.array([
    [56, 70], [81, 101], [119, 133],
    [22, 37], [103, 119], [56, 70],
    [81, 101], [119, 133], [22, 37],
    [103, 119], [56, 70], [81, 101],
    [119, 133], [22, 37], [103, 119]],
    dtype='float32')

inputs = torch.from_numpy(x_train)
targets = torch.from_numpy(y_train)

print(inputs.shape)
print(targets.shape)

torch.Size([15, 3])
torch.Size([15, 2])


In [3]:
train_ds = TensorDataset(inputs, targets) # pairs each input row with its matching target row

batch_size = 8
train_dl = DataLoader(train_ds, batch_size, shuffle=True) # splits the 15 samples into batches of 8 and shuffles their order every epoch

In [4]:
class LinearRegressionModel(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)   # takes the 3 raw inputs and projects them into 5 hidden values
        self.fc2 = nn.Linear(hidden_features, out_features)  # takes those 5 hidden values and maps them down to the 2 final outputs (apples, oranges)

    def forward(self, x):
        x = self.fc1(x)
        x = self.fc2(x)
        return x

model = LinearRegressionModel(in_features=3, hidden_features=5, out_features=2) # this method defines how data flows through the model when we call model(x)
print(model)

LinearRegressionModel(
  (fc1): Linear(in_features=3, out_features=5, bias=True)
  (fc2): Linear(in_features=5, out_features=2, bias=True)
)


In [5]:
criterion_mse = nn.MSELoss()

In [6]:
opt = torch.optim.SGD(model.parameters(), lr=1e-5)

In [7]:
def fit(num_epochs, model, loss_fn, opt, train_dl):
    for epoch in range(num_epochs):
        for xb, yb in train_dl:
            pred = model(xb)              # 1. Predict
            loss = loss_fn(pred, yb)      # 2. Calculate loss
            opt.zero_grad()               # 3. Reset old gradients
            loss.backward()               # 4. Compute new gradients
            opt.step()                    # 5. Update weights

        if (epoch+1) % 100 == 0:
            sys.stdout.write("\rEpoch [{}/{}], Loss: {:.4f}".format(epoch+1, num_epochs, loss.item()))

In [8]:
fit(1000, model, criterion_mse, opt, train_dl)

Epoch [200/1000], Loss: 1.6747

Epoch [400/1000], Loss: 1.1936

Epoch [600/1000], Loss: 0.7067

Epoch [800/1000], Loss: 0.7218

Epoch [1000/1000], Loss: 0.6042

In [9]:
preds = model(inputs)
final_loss = criterion_mse(preds, targets)

print("\n\nFinal Loss:", final_loss.item())
print("\nPredictions:\n", preds)
print("\nActual targets:\n", targets)



Final Loss: 0.5733102560043335

Predictions:
 tensor([[ 57.1109,  70.4866],
        [ 82.3446, 100.4163],
        [118.5075, 133.3984],
        [ 21.0775,  37.0741],
        [102.0931, 118.8177],
        [ 57.1109,  70.4866],
        [ 82.3446, 100.4163],
        [118.5075, 133.3984],
        [ 21.0775,  37.0741],
        [102.0931, 118.8177],
        [ 57.1109,  70.4866],
        [ 82.3446, 100.4163],
        [118.5075, 133.3984],
        [ 21.0775,  37.0741],
        [102.0931, 118.8177]], grad_fn=<AddmmBackward0>)

Actual targets:
 tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.],
        [ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.],
        [ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.]])
